# 03 — Demand estimation: logit, nested logit, random coefficients (with supply side and optimal instruments)

Mean utility of product $j$ (carrier × service type) in market $t$ (airport pair × quarter):

$$\delta_{jt} = -\alpha p_{jt} + \beta_1 \text{nonstop}_{jt} + \beta_2 \text{distance}_{jt} + \beta_3 \text{hub}_{jt}
  + \xi_{c(j)} + \xi_{r(t)} + \xi_{q(t)} + \Delta\xi_{jt}$$

with carrier, route (airport pair) and quarter fixed effects. Prices are average one-way fares in \$100 of 2025Q2;
*distance* is itinerary miles flown (000s); *hub* is the carrier's average share of departures at the endpoints.

| Model | Heterogeneity | Estimation |
|---|---|---|
| Logit | none | linear IV-GMM |
| Nested logit, nests = nonstop vs connecting (as specified) | within-service-type correlation $\rho$ | linear IV-GMM |
| Nested logit, one "air travel" nest vs the outside good | within-air-travel correlation $\rho$ | linear IV-GMM |
| Random coefficients (BLP) | normal RCs on the constant and nonstop; lognormal price coefficient | PyBLP GMM, demand only; then demand + supply; then optimal instruments |

**Instruments.** Section 1 tests every candidate instrument family. Hausman fares and engineering cost shifters
give upward-sloping demand, so the baseline uses BLP-style instruments: sums of rival characteristics, the
number of the carrier's own other products, numbers of rival carriers and products, and within-nest counts.
The cost shifters enter the marginal-cost equation of the supply side.

In [ ]:
%run 00_setup.ipynb

In [ ]:
%run 00_model_functions.ipynb

In [ ]:
RC_ROUTES = 120 if SMOKE else 600                       # routes in the random-coefficients estimation sample
STARTS = [(0.5, 0.3, 0.5), (1.0, 0.1, 1.0), (0.2, 0.6, 0.2)][: (1 if SMOKE else 3)]
IV = IV_SETS["baseline"]

p = pd.read_parquet(CLEAN / "products_airport_nondir.parquet")
p = add_instruments(p).sort_values(["market_ids", "carrier", "nonstop"]).reset_index(drop=True)
if SMOKE:
    keep = np.random.default_rng(0).choice(p.route.unique(), 400, replace=False)
    p = p[p.route.isin(keep)].reset_index(drop=True)
else:
    p.to_parquet(INTERIM / "products_airport_nondir_iv.parquet", index=False)
print(f"{len(p):,} products, {p.market_ids.nunique():,} markets, {p.route.nunique():,} routes")

## 1. Which instruments are valid? Logit and air-travel nested logit under each instrument family

The validity check is economic: demand must slope down. Hansen's J tests the overidentifying restrictions
(with ~335,000 observations it rejects almost any restriction at conventional levels, so the comparison
across sets is more informative than the p-value).

In [ ]:
diag_rows = []
for name, ivs in IV_SETS.items():
    lg = linear_iv(p, ivs)
    nl = linear_iv(p, ivs + (["iv_n_rival_products"] if "iv_n_rival_products" not in ivs else []), nest="air")
    diag_rows.append({"instruments": name, "logit alpha": -lg["alpha"], "(s.e.)": lg["alpha_se"],
                      "first-stage F": lg["first_stage"].get("prices", {}).get("F", np.nan), "J": lg["J"], "J df": lg["J_df"],
                      "NL alpha": -nl["alpha"], "NL rho": nl["rho"]})
# product fixed effects: identification from within-product changes over time only
pf = p.assign(product=p.route + ":" + p.carrier + ":" + p.nonstop.astype(str))
for name in ["baseline", "cost shifters only"]:
    lg = linear_iv(pf, IV_SETS[name], fe=("product", "quarter"), dummies=())
    diag_rows.append({"instruments": f"{name}, product + quarter FE", "logit alpha": -lg["alpha"], "(s.e.)": lg["alpha_se"],
                      "first-stage F": lg["first_stage"].get("prices", {}).get("F", np.nan), "J": lg["J"], "J df": lg["J_df"],
                      "NL alpha": np.nan, "NL rho": np.nan})
iv_diag = pd.DataFrame(diag_rows).set_index("instruments")
save_table(iv_diag, "iv_diagnostics", float_format="%.3f",
           note="Price sensitivity $\\alpha$ (utility per \\$100; positive means downward-sloping demand), route-clustered s.e., "
                "first-stage Wald F for price, Hansen J. Columns NL: nested logit with one air-travel nest. "
                "Route, quarter and carrier fixed effects unless stated.")
iv_diag

## 2. First stage (baseline instruments)

In [ ]:
nl_air = linear_iv(p, IV, nest="air")
nl_svc = linear_iv(p, IV, nest="service")
logit = linear_iv(p, IV)
fs = pd.DataFrame({
    "Price (logit)": logit["first_stage"]["prices"],
    "Price (NL air)": nl_air["first_stage"]["prices"],
    "log within-nest share (NL air)": nl_air["first_stage"]["lws"],
    "log within-nest share (NL nonstop/connecting)": nl_svc["first_stage"]["lws"],
}).T
save_table(fs, "first_stage", float_format="%.3f",
           note="Route-clustered first-stage Wald F statistics of the excluded instruments, partial $R^2$ and Shea's partial $R^2$ "
                "(which accounts for multiple endogenous regressors). Baseline instruments; route, quarter and carrier fixed effects.")
fs

In [ ]:
absorb = pd.DataFrame({"route": p.route.astype("category"), "quarter": p.quarter.astype("category"),
                       "carrier": p.carrier_fe.astype("category")})
fit = AbsorbingLS(p.prices, p[EXOG + IV].astype(float), absorb=absorb) \
    .fit(cov_type="clustered", clusters=p.route.astype("category").cat.codes)
fs_coef = pd.DataFrame({"coef": fit.params, "t": fit.tstats}).loc[IV]
save_table(fs_coef, "first_stage_coefficients", float_format="%.3f",
           note="Dependent variable: price (\\$100). Route, quarter and carrier fixed effects; route-clustered t statistics.")
fs_coef

## 3. Random coefficients

Estimated on a random sample of routes (all quarters of each sampled route); the random-coefficients
contraction has to be solved in every market at every trial value, so the full 100,000-market panel is used only
for evaluation at the estimates. Three starting values; the best is re-estimated with (a) the supply side and
(b) approximate optimal instruments (Conlon and Gortmaker 2020).

In [ ]:
rng = np.random.default_rng(20240918)
rc_routes = rng.choice(p.route.unique(), size=min(RC_ROUTES, p.route.nunique()), replace=False)
p_rc = p[p.route.isin(rc_routes)].reset_index(drop=True)
print(f"RC sample: {p_rc.route.nunique():,} routes, {p_rc.market_ids.nunique():,} markets, {len(p_rc):,} products")
prob_d = make_problem(p_rc, "rc", IV)
runs = []
with pyblp.parallel(4):
    for k, s0 in enumerate(STARTS):
        r = solve_rc(prob_d, "rc", sigma0=s0, method="1s")
        runs.append({"start": k, "sigma0": list(s0), **rc_params(r), "seconds": r.elapsed, "res": r})
        print(k, s0, round(runs[-1]["objective"], 4), np.round(runs[-1]["sigma"], 4), np.round(np.ravel(runs[-1]["pi"]), 4), round(r.elapsed))
best = min(runs, key=lambda d: d["objective"])
pd.DataFrame([{k: v for k, v in d.items() if k not in ("res", "beta")} for d in runs]).to_csv(RES / "rc_starting_values.csv", index=False)

In [ ]:
with pyblp.parallel(4):
    rc_d = solve_rc(prob_d, "rc", sigma0=best["sigma"], pi0=best["pi"], method="2s")
print(rc_d)

### Demand and supply jointly

Log marginal cost: $\ln c_{jt} = \gamma_1 \text{fuel}_{jt} + \gamma_2 \text{pilot}_{jt} + \gamma_3 \text{other cost}_{jt} + x_{jt}\gamma_x + \text{FE} + \omega_{jt}$,
with the Bertrand–Nash markup and route, quarter and carrier effects. Supply moments use the cost shifters and
the BLP instruments (which shift markups but are excluded from costs). Costs are bounded below at \$0.10.

In [ ]:
prob_s = make_problem(p_rc, "rc", IV, supply=True)
with pyblp.parallel(4):
    rc_s = solve_rc(prob_s, "rc", sigma0=np.diag(rc_d.sigma), pi0=rc_d.pi, method="2s", supply=True)
print(rc_s)

### Approximate optimal instruments

In [ ]:
with pyblp.parallel(4):
    oi = rc_s.compute_optimal_instruments(method="approximate")
    prob_o = oi.to_problem()
    rc_o = solve_rc(prob_o, "rc", sigma0=np.diag(rc_s.sigma), pi0=rc_s.pi, method="1s", supply=True)
print(rc_o)
# Sanity check: the optimal-instrument estimates are used only if they imply positive marginal costs for most products
mc_o = rc_o.compute_costs()[:, 0]
OI_OK = bool(np.mean(mc_o > 0) > 0.9)
print("optimal-IV estimates imply positive marginal cost for", round(float(np.mean(mc_o > 0)), 3), "of products; usable:", OI_OK)

## 4. Parameter estimates

In [ ]:
def lin_items(r, rho=False):
    items = [("Price sensitivity alpha", -r["alpha"], r["alpha_se"])]
    items += [(v, r["beta"][v], r["beta_se"][v]) for v in EXOG]
    if rho:
        items.append(("rho (nesting)", r["rho"], r["rho_se"]))
    return items

def rc_items(r):
    lab = dict(zip(r.beta_labels, range(len(r.beta_labels))))
    items = [("log median price sensitivity (pi_0)", float(r.pi[1, 0]), float(r.pi_se[1, 0]))]
    items += [(v, float(r.beta[lab[v], 0]), float(r.beta_se[lab[v], 0])) for v in EXOG]
    items += [("sigma: constant", float(r.sigma[0, 0]), float(r.sigma_se[0, 0])),
              ("sigma: log price coefficient", float(r.sigma[1, 1]), float(r.sigma_se[1, 1])),
              ("sigma: nonstop", float(r.sigma[2, 2]), float(r.sigma_se[2, 2]))]
    if r.gamma is not None and len(r.gamma):
        gl = dict(zip(r.gamma_labels, range(len(r.gamma_labels))))
        items += [(f"cost: {v}", float(r.gamma[gl[v], 0]), float(r.gamma_se[gl[v], 0])) for v in ["cs_fuel", "cs_pilot", "cs_exfuel"] if v in gl]
    return items

coef = coef_rows({"Logit": lin_items(logit), "NL (nonstop/connecting)": lin_items(nl_svc, True),
                  "NL (air travel)": lin_items(nl_air, True), "RC demand": rc_items(rc_d),
                  "RC demand+supply": rc_items(rc_s), "RC optimal IV": rc_items(rc_o)})
extra = pd.DataFrame({
    "Logit": [f"{len(p):,}", f"{logit['J']:.1f}"], "NL (nonstop/connecting)": [f"{len(p):,}", f"{nl_svc['J']:.1f}"],
    "NL (air travel)": [f"{len(p):,}", f"{nl_air['J']:.1f}"], "RC demand": [f"{len(p_rc):,}", f"{objective(rc_d):.1f}"],
    "RC demand+supply": [f"{len(p_rc):,}", f"{objective(rc_s):.1f}"], "RC optimal IV": [f"{len(p_rc):,}", f"{objective(rc_o):.1f}"]},
    index=["Products", "J / GMM objective"])
coef_out = pd.concat([coef, extra])
save_table(coef_out, "demand_estimates",
           note="Linear models: two-step IV-GMM with route fixed effects absorbed and quarter and carrier dummies. RC models: PyBLP, "
                f"{p_rc.route.nunique()} randomly sampled routes (all quarters), {RC_DRAWS} draws per market; the price coefficient is "
                "$-\\exp(\\pi_0+\\sigma_p\\nu)$. Route-clustered standard errors in parentheses.")
coef_out

## 5. Elasticities, diversion ratios, marginal costs and markups (full sample)

In [ ]:
PARAMS = {
    "Logit": ("logit", {"alpha": logit["alpha"]}),
    "NL (nonstop/connecting)": ("nested_service", {"alpha": nl_svc["alpha"], "rho": nl_svc["rho"]}),
    "NL (air travel)": ("nested_air", {"alpha": nl_air["alpha"], "rho": nl_air["rho"]}),
    "RC demand": ("rc", {"sigma": np.diag(rc_d.sigma).tolist(), "pi": rc_d.pi.tolist()}),
    "RC demand+supply": ("rc", {"sigma": np.diag(rc_s.sigma).tolist(), "pi": rc_s.pi.tolist()}),
    "RC optimal IV": ("rc", {"sigma": np.diag(rc_o.sigma).tolist(), "pi": rc_o.pi.tolist()}),
}
SLUG = {k: k.lower().replace(" ", "_").replace("/", "_").replace("(", "").replace(")", "").replace("+", "_") for k in PARAMS}
full = {}
post = {}
for name, (kind, prm) in PARAMS.items():
    with pyblp.parallel(4):
        full[name] = evaluate_fixed(p, kind, prm, IV)
    post[name] = post_estimation(full[name], p)
    if not SMOKE:
        post[name].to_parquet(INTERIM / f"post_{SLUG[name]}.parquet", index=False)
summ = pd.DataFrame({k: summarize_post(v) for k, v in post.items()})
save_table(summ, "elasticities_markups", float_format="%.3f",
           note="Full sample (all markets). Lerner index $(p-c)/p$ with marginal costs implied by static Bertrand--Nash pricing "
                "under carrier ownership. Diversion ratios: share of a product's lost passengers captured by the outside good / by "
                "the closest rival product after a small price increase.")
summ

In [ ]:
SHOW = ["Logit", "NL (air travel)", "RC demand+supply"]
by_c = []
for name in SHOW:
    pe = post[name].assign(cfe=lambda d: np.where(d.carrier.isin(MAJOR_CARRIERS), d.carrier, "OTH"))
    by_c.append(pe.groupby("cfe").apply(lambda d: pd.Series({
        f"{name}: elasticity": np.average(d.own_elasticity, weights=d.passengers),
        f"{name}: Lerner": np.median(d.lerner)})))
by_c = pd.concat(by_c, axis=1)
by_c.index = by_c.index.map(lambda c: CARRIER_NAMES.get(c, c))
save_table(by_c, "elasticities_by_carrier", float_format="%.2f",
           note="Passenger-weighted mean own-price elasticity and median Lerner index by ticketing carrier.")
by_c

### Markup plausibility checks

1. Lerner indices in (0, 1) and positive marginal costs.
2. Marginal cost should rise with miles flown and with the engineering cost shifters.
3. Benchmarks: airline price–cost margins in the structural literature typically lie between about 0.2 and 0.6
   (e.g. Berry and Jia 2010; Ciliberto and Williams 2014), and U.S. carriers' accounting operating margins (which
   also cover fixed and capacity costs) are typically 5–15%.

In [ ]:
checks = []
for name in post:
    d = post[name].join(p[["distance", "hub_presence", "cs_fuel", "cs_pilot", "carrier_fe"]])
    ab = pd.DataFrame({"quarter": d.quarter.astype("category"), "carrier": d.carrier_fe.astype("category")})
    f = AbsorbingLS(d.mc * 100, d[["distance", "nonstop", "hub_presence"]], absorb=ab) \
        .fit(cov_type="clustered", clusters=d.route.astype("category").cat.codes)
    c = np.corrcoef(d.mc, (d.cs_fuel + d.cs_pilot))[0, 1]
    checks.append({"model": name, "share Lerner in (0,1)": ((d.lerner > 0) & (d.lerner < 1)).mean(),
                   "share mc > 0": (d.mc > 0).mean(), "median mc ($)": d.mc.median() * 100,
                   "mc per 1,000 miles ($)": f.params["distance"], "(t)": f.tstats["distance"],
                   "corr(mc, engineering cost)": c})
checks = pd.DataFrame(checks).set_index("model")
save_table(checks, "markup_checks", float_format="%.3f",
           note="Marginal cost (\\$) regressed on miles flown, nonstop and hub presence with quarter and carrier fixed effects "
                "(route-clustered t). Last column: correlation of implied marginal cost with the fuel + pilot cost shifters.")
checks

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(10.5, 5.6))
for ax, name in zip(axes[0], SHOW):
    ax.hist(post[name].lerner.clip(-0.2, 1.2), bins=70, color=PALETTE[0], alpha=0.85)
    ax.set_title(name); ax.set_xlabel("Lerner index (p - c)/p")
for ax, name in zip(axes[1], SHOW):
    ax.hist(post[name].own_elasticity.clip(-15, 0), bins=70, color=PALETTE[1], alpha=0.85)
    ax.set_xlabel("Own-price elasticity")
axes[0, 0].set_ylabel("Products"); axes[1, 0].set_ylabel("Products")
fig.tight_layout(); savefig(fig, "elasticity_lerner_distribution")

### Diversion ratios on illustrative routes (latest quarter)

In [ ]:
def diversion_matrix(res, p, market):
    idx = p.index[p.market_ids == market]
    D = res.compute_diversion_ratios(market_id=market)
    labels = [f"{c}{' NS' if n else ' 1-stop'}" for c, n in zip(p.loc[idx, "carrier"], p.loc[idx, "nonstop"])]
    M = pd.DataFrame(D[:, :len(idx)], index=labels, columns=labels)
    out = M.copy()
    for i in range(len(labels)):
        out.iloc[i, i] = np.nan
    out["Outside"] = np.diag(M.values)
    return out

for route in ["HNL-SEA", "FLL-LGA", "EWR-FLL", "BOS-FLL"]:
    cand = p[p.route == route]
    if not len(cand):
        continue
    mk = cand.market_ids[cand.quarter == cand.quarter.max()].iat[0]
    for name in ["NL (air travel)", "RC demand+supply"]:
        M = diversion_matrix(full[name], p, mk)
        save_table(M, f"diversion_{route}_{name.split()[0].lower()}", float_format="%.3f",
                   note=f"Diversion ratios, {mk}, {name}. Row product raises its price; entries are the share of its lost "
                        "passengers going to the column product or to the outside good.")
        print(mk, name); display(M.round(3))

In [ ]:
if True:
    save_json({
        "linear": {k: {kk: vv for kk, vv in v.items() if kk != "res"} for k, v in
                   {"logit": logit, "nested_service": nl_svc, "nested_air": nl_air}.items()},
        "rc_demand": rc_params(rc_d), "rc_supply": rc_params(rc_s), "rc_optimal": {**rc_params(rc_o), "usable": OI_OK},
        "rc_routes": list(map(str, rc_routes)), "summary": summ.to_dict(), "first_stage": fs.to_dict(),
        "iv_diagnostics": iv_diag.reset_index().to_dict("records"),
    }, "demand_full_sample")